<a href="https://colab.research.google.com/github/krishuynh2222/retail-end-to-end-analytics/blob/main/01_notebooks/03_quality_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03: Quality Check
##### **PURPOSE:**
###### Verify that every fix made in 02_data_cleaning.ipynb actually worked.
###### Run this notebook after cleaning is complete.

###### **HOW CHECKS ARE CATEGORIZED:**
###### PASS   The fix worked. This is what we expect.
###### FAIL   Something is still wrong. Fix the cleaning cell and re-run.
###### WARN   A known issue that was documented in cleaning.

###### **EACH CHECK EXPLAINS:**
######   WHAT   what is being verified
######   WHY    why this matters for downstream analysis or JOIN accuracy
######   FIX    which cleaning cell to look at if the check fails

###### HOW TO USE THIS NOTEBOOK
######   Run all cells from top to bottom.
######   If you see FAIL, go to 02_data_cleaning.ipynb, fix that table's cell,re-run that single cell, then come back and re-run this entire notebook.

##### **OUTPUT**
###### audit/03_qc_results.csv   full list of all checks with status


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

CLEAN_DIR = '/content/drive/MyDrive/AmazinChoices/datasets_clean'
AUDIT_DIR = '/content/drive/MyDrive/AmazinChoices/audit'

os.makedirs(AUDIT_DIR, exist_ok=True)

print("CLEAN_DIR: " + CLEAN_DIR)
print("AUDIT_DIR: " + AUDIT_DIR)


CLEAN_DIR: /content/drive/MyDrive/AmazinChoices/datasets_clean
AUDIT_DIR: /content/drive/MyDrive/AmazinChoices/audit


In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

## **Define the QC engine**
###### All quality checks in this notebook go through the check() function.
###### This ensures every result is recorded consistently and the summary
###### at the end gives accurate counts.

###### **check() takes five arguments:**
###### - table string which table this check belongs to
###### - rule string short description of what is being verified
###### - passed bool True = PASS, False = FAIL or WARN
###### - detail string  explains what went wrong (shown only when check fails)
###### - severity   string  'FAIL' for blocking issues, 'WARN' for known issues

In [ ]:
all_qc_results = []


def check(table, rule, passed, detail='', severity='FAIL'):
    """
    Record one quality check result and print it to the screen.
    Results are stored in all_qc_results for the summary at the end.
    """
    if passed:
        status = 'PASS'
    elif severity == 'WARN':
        status = 'WARN'
    else:
        status = 'FAIL'

    all_qc_results.append({
        'table':    table,
        'rule':     rule,
        'status':   status,
        'detail':   detail,
    })

    if passed:
        print("  PASS   " + rule)
    elif severity == 'WARN':
        print("  WARN   " + rule)
        print("         " + detail)
    else:
        print("  FAIL   " + rule)
        print("         " + detail)


def section_header(title, description):
    """Print a section header to make the output easier to read."""
    print("")
    print("=" * 60)
    print(title)
    print(description)
    print("=" * 60)


def load_clean_table(table_name):
    """Load one clean CSV. Return None if the file does not exist."""
    file_path = os.path.join(CLEAN_DIR, table_name + '.csv')
    if not os.path.exists(file_path):
        print("  FILE NOT FOUND: " + file_path)
        return None
    return pd.read_csv(file_path)


print("QC engine defined. Ready to run checks.")


QC engine defined. Ready to run checks.


## Load all 11 clean tables
######  We load all tables once here and reuse them throughout the notebook.
######  This is faster than reading each CSV file multiple times.

In [ ]:
print("Loading all 11 clean tables...")

orders    = load_clean_table('orders')
items     = load_clean_table('order_items')
customers = load_clean_table('customer_profiles')
refunds   = load_clean_table('refunds')
costs     = load_clean_table('cost_history')
products  = load_clean_table('product_master')
promos    = load_clean_table('promotions_log')
po        = load_clean_table('purchase_orders')
mkt       = load_clean_table('marketing_spend')
interacts = load_clean_table('customer_interactions')
shipping  = load_clean_table('shipping_rates')

loaded_count = sum(1 for t in [orders, items, customers, refunds, costs,
                                products, promos, po, mkt, interacts, shipping]
                   if t is not None)
print("")
print(str(loaded_count) + " of 11 tables loaded.")
if loaded_count < 11:
    print("Some tables could not be loaded. Run 02_data_cleaning.ipynb first.")

Loading all 11 clean tables...

11 of 11 tables loaded.



## Check 1 - File existence

###### Verify that all 11 clean CSV files were saved to CLEAN_DIR.
###### This catches the case where a cleaning cell ran but the .to_csv() line failed.
###### FIX if this fails: re-run the missing table's cell in 02_data_cleaning.ipynb.

In [ ]:
section_header(
    "CHECK: FILE EXISTENCE",
    "All 11 cleaned CSV files must exist in CLEAN_DIR before BigQuery upload."
)

ALL_TABLE_NAMES = [
    'orders', 'order_items', 'customer_profiles', 'customer_interactions',
    'marketing_spend', 'refunds', 'cost_history', 'product_master',
    'promotions_log', 'purchase_orders', 'shipping_rates'
]

for table_name in ALL_TABLE_NAMES:
    file_path  = os.path.join(CLEAN_DIR, table_name + '.csv')
    file_exists = os.path.exists(file_path)
    if file_exists:
        size_mb = round(os.path.getsize(file_path) / 1024 / 1024, 1)
        check(table_name, table_name + '.csv exists (' + str(size_mb) + ' MB)', True)
    else:
        check(table_name, table_name + '.csv exists', False,
              'File not found at ' + file_path + '. Re-run the cleaning cell for this table.')



CHECK: FILE EXISTENCE
All 11 cleaned CSV files must exist in CLEAN_DIR before BigQuery upload.
  PASS   orders.csv exists (15.0 MB)
  PASS   order_items.csv exists (13.7 MB)
  PASS   customer_profiles.csv exists (5.5 MB)
  PASS   customer_interactions.csv exists (10.7 MB)
  PASS   marketing_spend.csv exists (6.4 MB)
  PASS   refunds.csv exists (7.1 MB)
  PASS   cost_history.csv exists (5.9 MB)
  PASS   product_master.csv exists (4.7 MB)
  PASS   promotions_log.csv exists (6.6 MB)
  PASS   purchase_orders.csv exists (9.4 MB)
  PASS   shipping_rates.csv exists (3.0 MB)


## Orders table

###### Each check in this section corresponds to one fix made in CELL 5 of 02_data_cleaning.ipynb.

###### If any check here fails, go to Cell 5 of the cleaning notebook and look for the fix labeled with the same problem description.

In [ ]:
section_header(
    "CHECK: ORDERS",
    "Email JOIN key, dates, currency, tax and shipping fee, promo code."
)

if orders is not None:

    # CHECK: customer_email has no uppercase letters
    # WHAT: verify that normalize_email() converted all emails to lowercase
    # WHY:  if any email is uppercase, it will not match customer_profiles.customer_email
    #       in a JOIN, causing that customer's rows to be silently dropped

    uppercase_email_count = orders['customer_email'].str.contains('[A-Z]', na=False).sum()
    check(
        'orders',
        'customer_email: no uppercase letters (JOIN key to customer_profiles)',
        uppercase_email_count == 0,
        str(uppercase_email_count) + ' email addresses still contain uppercase letters. '
        'Fix: re-run Cell 5 in 02_data_cleaning.ipynb'
    )

    # CHECK: customer_email matches the x@x.x format
    # WHAT: basic format validation to catch completely malformed emails
    # WHY:  a value like 'N/A' or '-' in the email column will silently
    #       match nothing in a JOIN, losing revenue and customer data

    invalid_email_count = (~orders['customer_email'].str.match(r'^[^@]+@[^@]+\.[^@]+$', na=True)).sum()
    check(
        'orders',
        'customer_email: all values match the x@x.x format',
        invalid_email_count == 0,
        str(invalid_email_count) + ' email values do not look like valid email addresses.'
    )

    # CHECK: order_date has no NaT values
    # WHAT: verify that parse_date() successfully converted all timestamps
    # WHY:  NaT in order_date breaks all time-series analysis.
    #       Monthly revenue charts, year-over-year comparisons, and cohort
    #       analysis all depend on order_date being a valid date.

    nat_date_count = pd.to_datetime(orders['order_date'], errors='coerce').isna().sum()
    check(
        'orders',
        'order_date: zero NaT values (all mixed date formats parsed successfully)',
        nat_date_count == 0,
        str(nat_date_count) + ' order_date values are NaT. '
        'Fix: check parse_date() in Cell 5 of 02_data_cleaning.ipynb'
    )

    # CHECK: order_year_month follows YYYY-MM format
    # WHAT: verify the format used for monthly grouping is consistent
    # WHY:  if some rows have 'Jan 2025' and others have '2025-01',
    #       a GROUP BY will create two separate buckets for the same month

    bad_yearmonth_count = (~orders['order_year_month'].str.match(r'^\d{4}-\d{2}$', na=False)).sum()
    check(
        'orders',
        'order_year_month: all values are in YYYY-MM format',
        bad_yearmonth_count == 0,
        str(bad_yearmonth_count) + ' values do not match YYYY-MM format.'
    )

    # CHECK: currency column contains only 'USD'
    # WHAT: verify that all currency variants were normalized
    # WHY:  variants like 'Usd', 'US D', '$' would create separate GROUP BY
    #       buckets, making currency look like 3 or 4 different currencies
    #       instead of one. Every currency metric would be wrong.

    unique_currencies = orders['currency'].dropna().unique().tolist()
    check(
        'orders',
        "currency: every row contains exactly 'USD' (all variants normalized)",
        unique_currencies == ['USD'],
        'Non-USD values still present: ' + str(unique_currencies) + '. '
        "Fix: re-run Cell 5 and check the out['currency'] = 'USD' line"
    )

    # CHECK: shipping_fee has no negative values
    # WHAT: verify that .clip(lower=0) removed all negative shipping fees
    # WHY:  a negative shipping fee would subtract from order revenue when
    #       added to totals, making revenue look smaller than it really is

    neg_fee_count = (orders['shipping_fee'] < 0).sum()
    check(
        'orders',
        'shipping_fee: no negative values (clipped to 0 during cleaning)',
        neg_fee_count == 0,
        str(neg_fee_count) + ' rows still have negative shipping_fee. '
        'Fix: re-run Cell 5 and check the .clip(lower=0) call on shipping_fee'
    )

    # CHECK: tax has no negative values
    # WHAT: verify that .clip(lower=0) removed all negative tax values
    # WHY:  negative tax would inflate net revenue incorrectly

    neg_tax_count = (orders['tax'] < 0).sum()
    check(
        'orders',
        'tax: no negative values (clipped to 0 during cleaning)',
        neg_tax_count == 0,
        str(neg_tax_count) + ' rows still have negative tax values. '
        'Fix: re-run Cell 5 and check the .clip(lower=0) call on tax'
    )

    # CHECK: promo_code has no 'NONE' string and no whitespace
    # WHAT: verify that 'NONE' was replaced with NaN and whitespace was stripped
    # WHY:  if 'NONE' stays as a string, GROUP BY will count it as a real promo
    #       code, making it look like there is a promotion called NONE.
    #       Whitespace makes 'PROMO10 ' and 'PROMO10' look like different codes.

    if orders['promo_code'].notna().any():
        has_none_string  = (orders['promo_code'].dropna() == 'NONE').any()
        has_whitespace   = orders['promo_code'].dropna().str.contains(r'\s', regex=True).any()
        check(
            'orders',
            "promo_code: no 'NONE' string, no whitespace",
            not has_none_string and not has_whitespace,
            "NONE string present: " + str(has_none_string) + ", whitespace present: " + str(has_whitespace)
        )

    # CHECK: order_status contains only expected values in Title Case
    # WHAT: verify that .str.title() was applied and no unexpected values exist
    # WHY:  if both 'Completed' and 'completed' exist, they are two separate
    #       categories in a GROUP BY, splitting data that should be together

    expected_statuses = {'Completed', 'Shipped', 'Cancelled', 'Pending', 'Refunded'}
    actual_statuses   = set(orders['order_status'].dropna().unique())
    unexpected_statuses = actual_statuses - expected_statuses
    check(
        'orders',
        'order_status: only known values, all in Title Case',
        len(unexpected_statuses) == 0,
        'Unexpected values found: ' + str(unexpected_statuses)
    )

    # CHECK: order_id has no duplicates
    # WHAT: verify that orders is a proper fact table with one row per order
    # WHY:  duplicate order_ids cause double-counting in every revenue metric.
    #       Total revenue, order count, and AOV will all be inflated.

    duplicate_order_count = orders['order_id'].duplicated().sum()
    check(
        'orders',
        'order_id: no duplicate values (one row per order)',
        duplicate_order_count == 0,
        str(duplicate_order_count) + ' duplicate order_ids found. This inflates all revenue metrics.'
    )

    # CHECK: no _raw columns remain
    # WHAT: verify that all original raw columns were dropped after cleaning
    # WHY:  if raw columns remain in the clean file, downstream tools may
    #       accidentally use the uncleaned values instead of the clean ones.
    #       It also wastes storage in BigQuery.

    remaining_raw_cols = [c for c in orders.columns if c.endswith('_raw')]
    check(
        'orders',
        'no _raw columns remaining in clean file',
        len(remaining_raw_cols) == 0,
        'These _raw columns were not dropped: ' + str(remaining_raw_cols)
    )

    print("")
    print("  Summary:")
    print("  Rows: " + str(len(orders)))
    print("  shipping_fee null: " + str(round(orders['shipping_fee'].isna().mean() * 100, 1)) + "% (expected, some orders have free shipping)")
    print("  promo_code null:   " + str(round(orders['promo_code'].isna().mean() * 100, 1)) + "% (expected, not every order uses a promo code)")


CHECK: ORDERS
Email JOIN key, dates, currency, tax and shipping fee, promo code.
  PASS   customer_email: no uppercase letters (JOIN key to customer_profiles)
  PASS   customer_email: all values match the x@x.x format
  PASS   order_date: zero NaT values (all mixed date formats parsed successfully)
  PASS   order_year_month: all values are in YYYY-MM format
  PASS   currency: every row contains exactly 'USD' (all variants normalized)
  PASS   shipping_fee: no negative values (clipped to 0 during cleaning)
  PASS   tax: no negative values (clipped to 0 during cleaning)
  PASS   promo_code: no 'NONE' string, no whitespace
  PASS   order_status: only known values, all in Title Case
  PASS   order_id: no duplicate values (one row per order)
  PASS   no _raw columns remaining in clean file

  Summary:
  Rows: 110000
  shipping_fee null: 11.9% (expected, some orders have free shipping)
  promo_code null:   32.6% (expected, not every order uses a promo code)


## Order_items table

In [ ]:
section_header(
    "CHECK: ORDER_ITEMS",
    "Revenue formula correctness, price and qty constraints, SKU format, FK to orders."
)

if items is not None:

    # CHECK: item_price has no negative values
    # WHAT: verify .clip(lower=0) was applied
    # WHY:  negative item_price makes line_revenue negative, corrupting total revenue

    neg_price_count = (items['item_price'] < 0).sum()
    check(
        'order_items',
        'item_price: no negative values (clipped to 0)',
        neg_price_count == 0,
        str(neg_price_count) + ' rows have negative item_price'
    )

    # CHECK: discount_amt has no negative values
    # WHY:  a negative discount means you are charging the customer extra,
    #       not giving them a discount. This is almost always a data entry error.
    neg_disc_count = (items['discount_amt'] < 0).sum()
    check(
        'order_items',
        'discount_amt: no negative values (clipped to 0)',
        neg_disc_count == 0,
        str(neg_disc_count) + ' rows have negative discount_amt'
    )

    # CHECK: discount_amt does not exceed item_price
    # WHAT: verify the cap was applied (discount cannot be bigger than the price)
    # WHY:  if discount > price then (price - discount) is negative,
    #       making line_revenue negative and reducing total revenue incorrectly

    excess_discount_count = (items['discount_amt'].fillna(0) > items['item_price'].fillna(0)).sum()
    check(
        'order_items',
        'discount_amt is less than or equal to item_price on every row',
        excess_discount_count == 0,
        str(excess_discount_count) + ' rows still have discount_amt > item_price',
        severity='WARN'
    )

    # CHECK: qty is positive on all rows
    # WHAT: verify that zero and negative qty values were set to NaN
    # WHY:  qty is the multiplier in line_revenue = (price - discount) * qty.
    #       A qty of 0 makes line_revenue 0. A negative qty makes it negative.
    #       Both destroy the accuracy of total revenue.
    non_null_qty  = items['qty'].dropna()
    bad_qty_count = (non_null_qty <= 0).sum()
    check(
        'order_items',
        'qty: all values are greater than zero (zero and negative set to NaN)',
        bad_qty_count == 0,
        str(bad_qty_count) + ' rows still have qty that is zero or negative'
    )

    # CHECK: line_revenue formula is correct
    # WHAT: verify that line_revenue = (item_price - discount_amt) * qty on every row
    # WHY:  this is the most important check in the entire pipeline.
    #       If the formula is wrong, every downstream revenue metric is wrong:
    #       total revenue, gross margin, net profit, category performance, etc.
    rows_with_all_values = items.dropna(subset=['item_price', 'qty', 'line_revenue'])
    expected_revenue = (rows_with_all_values['item_price'] - rows_with_all_values['discount_amt'].fillna(0)) * rows_with_all_values['qty']
    formula_mismatch_count = (rows_with_all_values['line_revenue'] - expected_revenue).abs().gt(0.01).sum()
    check(
        'order_items',
        'line_revenue equals (item_price minus discount_amt) times qty on every row',
        formula_mismatch_count == 0,
        str(formula_mismatch_count) + ' rows have line_revenue that does not match the formula result'
    )

    # CHECK: sku is uppercase with no whitespace
    # WHAT: verify .str.upper().str.strip() was applied
    # WHY:  sku is the JOIN key to product_master and cost_history.
    #       'IGN-TEA-001' and 'ign-tea-001' will not match in a JOIN.
    #       You lose cost data for those products, making gross margin wrong.
    sku_has_lowercase  = items['sku'].dropna().str.contains('[a-z]').any()
    sku_has_whitespace = items['sku'].dropna().str.contains(r'\s', regex=True).any()
    check(
        'order_items',
        'sku: all values are uppercase with no whitespace (JOIN key to product_master)',
        not sku_has_lowercase and not sku_has_whitespace,
        'lowercase present: ' + str(sku_has_lowercase) + ', whitespace present: ' + str(sku_has_whitespace)
    )

    # CHECK: every order_id in order_items exists in orders
    # WHAT: verify referential integrity of the foreign key
    # WHY:  if order_items has order_ids that do not exist in orders,
    #       those rows cannot be enriched with order-level data (platform,
    #       customer email, order date). They become orphan records.

    if orders is not None:
        orphan_line_count = (~items['order_id'].isin(orders['order_id'])).sum()
        check(
            'order_items',
            'every order_id in order_items exists in orders (foreign key check)',
            orphan_line_count == 0,
            str(orphan_line_count) + ' order_item rows reference order_ids that do not exist in orders',
            severity='WARN'
        )

    remaining_raw_cols = [c for c in items.columns if c.endswith('_raw')]
    check(
        'order_items',
        'no _raw columns remaining in clean file',
        len(remaining_raw_cols) == 0,
        'These _raw columns were not dropped: ' + str(remaining_raw_cols)
    )

    print("")
    print("  Summary:")
    print("  Rows: " + str(len(items)))
    print("  Total line_revenue: $" + str(round(items['line_revenue'].sum())))
    print("  item_price null: " + str(round(items['item_price'].isna().mean() * 100, 1)) + "%")
    print("  qty null:        " + str(round(items['qty'].isna().mean() * 100, 1)) + "%")
    print("  sku null:        " + str(round(items['sku'].isna().mean() * 100, 1)) + "%")


CHECK: ORDER_ITEMS
Revenue formula correctness, price and qty constraints, SKU format, FK to orders.
  PASS   item_price: no negative values (clipped to 0)
  PASS   discount_amt: no negative values (clipped to 0)
  PASS   discount_amt is less than or equal to item_price on every row
  PASS   qty: all values are greater than zero (zero and negative set to NaN)
  PASS   line_revenue equals (item_price minus discount_amt) times qty on every row
  PASS   sku: all values are uppercase with no whitespace (JOIN key to product_master)
  PASS   every order_id in order_items exists in orders (foreign key check)
  PASS   no _raw columns remaining in clean file

  Summary:
  Rows: 130000
  Total line_revenue: $5413551
  item_price null: 5.9%
  qty null:        6.0%
  sku null:        7.0%


## customer_profiles table

In [ ]:
section_header(
    "CHECK: CUSTOMER_PROFILES",
    "Email JOIN key, deduplication, state format, Wholesale? flag."
)

if customers is not None:

    uppercase_email_count = customers['customer_email'].str.contains('[A-Z]', na=False).sum()
    check(
        'customer_profiles',
        'customer_email: no uppercase letters (JOIN key to orders)',
        uppercase_email_count == 0,
        str(uppercase_email_count) + ' emails still have uppercase letters'
    )

    duplicate_email_count = customers['customer_email'].duplicated().sum()
    check(
        'customer_profiles',
        'customer_email: no duplicate values (one row per customer)',
        duplicate_email_count == 0,
        str(duplicate_email_count) + ' duplicate emails found. '
        'Fix: re-run Cell 7 in 02_data_cleaning.ipynb'
    )

    state_has_lowercase = customers['state'].dropna().str.contains('[a-z]').any()
    check(
        'customer_profiles',
        'state: all values are uppercase (standard format for US state codes)',
        not state_has_lowercase,
        'Some state values still contain lowercase letters'
    )

    nat_date_pct = pd.to_datetime(customers['first_purchase_date'], errors='coerce').isna().mean() * 100
    check(
        'customer_profiles',
        'first_purchase_date: null percentage is 40% or less (audit found 37.5% unparseable in raw)',
        nat_date_pct <= 40,
        str(round(nat_date_pct, 1)) + '% are NaT. Higher than the 37.5% found in the audit.'
    )

    # NOTE: Wholesale? rows are intentionally kept as-is
    # This is a WARN, not a FAIL. The rows are correctly preserved.
    # A business person must decide what to do with these customers.
    wholesale_count = (customers['segment'].str.strip() == 'Wholesale?').sum()
    check(
        'customer_profiles',
        "segment='Wholesale?' rows are retained (not silently fixed, awaiting business decision)",
        True,
        str(wholesale_count) + ' rows have segment = Wholesale?. '
        'Ask the business: real wholesale customers or data entry mistake?',
        severity='WARN'
    )
    if wholesale_count > 0:
        print("         " + str(wholesale_count) + " Wholesale? rows are present and waiting for business confirmation.")

    print("")
    print("  Summary:")
    print("  Rows: " + str(len(customers)))
    print("  Segment breakdown: " + str(customers['segment'].value_counts().to_dict()))


CHECK: CUSTOMER_PROFILES
Email JOIN key, deduplication, state format, Wholesale? flag.
  PASS   customer_email: no uppercase letters (JOIN key to orders)
  PASS   customer_email: no duplicate values (one row per customer)
  PASS   state: all values are uppercase (standard format for US state codes)
  PASS   first_purchase_date: null percentage is 40% or less (audit found 37.5% unparseable in raw)
  PASS   segment='Wholesale?' rows are retained (not silently fixed, awaiting business decision)
         15885 Wholesale? rows are present and waiting for business confirmation.

  Summary:
  Rows: 80000
  Segment breakdown: {'New': 16097, 'Churn Risk': 16038, 'Returning': 16020, 'VIP': 15960, 'Wholesale?': 15885}


## Refunds table

In [ ]:
section_header(
    "CHECK: REFUNDS",
    "Amount constraints, reason casing, empty string handling, FK to orders."
)

if refunds is not None:

    neg_refund_count = (refunds['refund_amount'] < 0).sum()
    check(
        'refunds',
        'refund_amount: no negative values',
        neg_refund_count == 0,
        str(neg_refund_count) + ' rows have negative refund_amount'
    )

    # CHECK: no zero-dollar refunds
    # WARN not FAIL because some systems generate $0 refund records as acknowledgements
    zero_refund_count = (refunds['refund_amount'].dropna() == 0).sum()
    check(
        'refunds',
        'refund_amount: no $0 values (investigate if any exist)',
        zero_refund_count == 0,
        str(zero_refund_count) + ' rows have $0 refund. Could be reversed payments or test data.',
        severity='WARN'
    )

    # CHECK: no empty strings in reason
    # WHAT: verify that '' was replaced with NaN
    # WHY:  an empty string in reason creates a blank GROUP BY bucket that
    #       inflates the count for 'no reason given' and hides the real breakdown
    empty_reason_count = (refunds['reason'].dropna() == '').sum()
    check(
        'refunds',
        "reason: no empty strings (empty strings were replaced with NaN)",
        empty_reason_count == 0,
        str(empty_reason_count) + ' rows still have empty string as reason value. '
        'Fix: re-run Cell 9 in 02_data_cleaning.ipynb'
    )

    # CHECK: reason values are in Title Case with known values only
    # WHAT: verify .str.title() was applied and no typos or unknown values remain
    # WHY:  without Title Case, 'damaged', 'Damaged', and 'DAMAGED' are three
    #       separate categories in a GROUP BY. This splits real data into fragments.
    expected_reasons  = {'Other', 'Customer Changed Mind', 'Missing Items',
                         'Damaged', 'Late Delivery', 'Wrong Item'}
    actual_reasons    = set(refunds['reason'].dropna().unique())
    unexpected_reasons = actual_reasons - expected_reasons
    check(
        'refunds',
        'reason: Title Case applied, only known reason values exist',
        len(unexpected_reasons) == 0,
        'Unexpected reason values: ' + str(unexpected_reasons) + '. '
        'Check whether .str.title() was applied correctly in Cell 9.',
        severity='WARN'
    )

    if orders is not None:
        orphan_refund_count = (~refunds['order_id'].isin(orders['order_id'])).sum()
        check(
            'refunds',
            'every order_id in refunds exists in orders (foreign key check)',
            orphan_refund_count == 0,
            str(orphan_refund_count) + ' refund rows reference order_ids not found in orders',
            severity='WARN'
        )

    remaining_raw_cols = [c for c in refunds.columns if c.endswith('_raw')]
    check(
        'refunds',
        'no _raw columns remaining in clean file',
        len(remaining_raw_cols) == 0,
        'These _raw columns were not dropped: ' + str(remaining_raw_cols)
    )

    print("")
    print("  Summary:")
    print("  Rows: " + str(len(refunds)))
    print("  Total refunded: $" + str(round(refunds['refund_amount'].sum())))
    print("  Average refund: $" + str(round(refunds['refund_amount'].mean(), 2)))
    print("  Reason breakdown: " + str(refunds['reason'].value_counts().to_dict()))


CHECK: REFUNDS
Amount constraints, reason casing, empty string handling, FK to orders.
  PASS   refund_amount: no negative values
  PASS   refund_amount: no $0 values (investigate if any exist)
  PASS   reason: no empty strings (empty strings were replaced with NaN)
  PASS   reason: Title Case applied, only known reason values exist
  PASS   every order_id in refunds exists in orders (foreign key check)
  PASS   no _raw columns remaining in clean file

  Summary:
  Rows: 80000
  Total refunded: $1693717
  Average refund: $23.54
  Reason breakdown: {'Other': 12731, 'Customer Changed Mind': 12648, 'Missing Items': 12545, 'Damaged': 12494, 'Late Delivery': 12461, 'Wrong Item': 12375}


## cost_history table

In [ ]:
section_header(
    "CHECK GROUP 6: COST_HISTORY",
    "total_cost formula correctness, no negative costs, SKU format."
)

if costs is not None:

    nat_date_count = pd.to_datetime(costs['effective_date'], errors='coerce').isna().sum()
    check(
        'cost_history',
        'effective_date: zero NaT values (all dates parsed)',
        nat_date_count == 0,
        str(nat_date_count) + ' effective_date values are NaT'
    )

    sku_has_lowercase  = costs['sku'].dropna().str.contains('[a-z]').any()
    sku_has_whitespace = costs['sku'].dropna().str.contains(r'\s', regex=True).any()
    check(
        'cost_history',
        'sku: uppercase with no whitespace (JOIN key to order_items and product_master)',
        not sku_has_lowercase and not sku_has_whitespace,
        'lowercase: ' + str(sku_has_lowercase) + ', whitespace: ' + str(sku_has_whitespace)
    )

    # CHECK: total_cost has no null values
    # WHAT: verify that fillna(0) was used when computing total_cost
    # WHY:  total_cost is used in gross margin calculations.
    #       If total_cost is null for a row, the gross margin for that product
    #       will also be null, making category-level margin reports incomplete.
    null_total_cost_count = costs['total_cost'].isna().sum()
    check(
        'cost_history',
        'total_cost: zero null values (fillna(0) was used in formula)',
        null_total_cost_count == 0,
        str(null_total_cost_count) + ' null values in total_cost'
    )

    neg_total_cost_count = (costs['total_cost'] < 0).sum()
    check(
        'cost_history',
        'total_cost: no negative values (abs() was applied to base_cost and packaging_cost)',
        neg_total_cost_count == 0,
        str(neg_total_cost_count) + ' negative total_cost values remain'
    )

    # CHECK: total_cost formula is correct
    # WHAT: verify total_cost = base_cost + packaging_cost on every row
    # WHY:  if the formula is wrong, every gross margin metric is wrong.
    #       A $1 error in COGS on 50,000 rows = $50,000 error in total COGS.
    rows_with_costs = costs.dropna(subset=['base_cost', 'packaging_cost', 'total_cost'])
    expected_cost   = rows_with_costs['base_cost'] + rows_with_costs['packaging_cost']
    formula_mismatch_count = (rows_with_costs['total_cost'] - expected_cost).abs().gt(0.001).sum()
    check(
        'cost_history',
        'total_cost equals base_cost plus packaging_cost on every row',
        formula_mismatch_count == 0,
        str(formula_mismatch_count) + ' rows have total_cost that does not match the formula'
    )

    remaining_raw_cols = [c for c in costs.columns if c.endswith('_raw')]
    check(
        'cost_history',
        'no _raw columns remaining in clean file',
        len(remaining_raw_cols) == 0,
        'These _raw columns were not dropped: ' + str(remaining_raw_cols)
    )

    print("")
    print("  Summary:")
    print("  Rows: " + str(len(costs)))
    print("  Unique SKUs: " + str(costs['sku'].nunique()))
    print("  Average total_cost: $" + str(round(costs['total_cost'].mean(), 2)))
    print("  base_cost null:      " + str(round(costs['base_cost'].isna().mean() * 100, 1)) + "% (expected around 10%)")
    print("  packaging_cost null: " + str(round(costs['packaging_cost'].isna().mean() * 100, 1)) + "% (expected around 12%)")


CHECK GROUP 6: COST_HISTORY
total_cost formula correctness, no negative costs, SKU format.
  PASS   effective_date: zero NaT values (all dates parsed)
  PASS   sku: uppercase with no whitespace (JOIN key to order_items and product_master)
  PASS   total_cost: zero null values (fillna(0) was used in formula)
  PASS   total_cost: no negative values (abs() was applied to base_cost and packaging_cost)
  PASS   total_cost equals base_cost plus packaging_cost on every row
  PASS   no _raw columns remaining in clean file

  Summary:
  Rows: 100000
  Unique SKUs: 56140
  Average total_cost: $4.94
  base_cost null:      9.8% (expected around 10%)
  packaging_cost null: 11.9% (expected around 12%)


## product_master table

In [ ]:
section_header(
    "CHECK GROUP 7: PRODUCT_MASTER",
    "SKU format, no duplicates, Title Case, launch_date null rate."
)

if products is not None:

    sku_has_lowercase  = products['sku'].str.contains('[a-z]').any()
    sku_has_whitespace = products['sku'].str.contains(r'\s', regex=True).any()
    sku_has_duplicates = products['sku'].duplicated().sum()
    check(
        'product_master',
        'sku: uppercase, no whitespace, no duplicates (dimension table key)',
        not sku_has_lowercase and not sku_has_whitespace and sku_has_duplicates == 0,
        'lowercase: ' + str(sku_has_lowercase) +
        ', whitespace: ' + str(sku_has_whitespace) +
        ', duplicates: ' + str(sku_has_duplicates)
    )

    product_name_all_caps = products['product_name'].str.isupper().any()
    check(
        'product_master',
        'product_name: no ALL CAPS values (Title Case applied)',
        not product_name_all_caps,
        'Some product_name values are still ALL CAPS. Fix: re-run Cell 11 in 02_data_cleaning.ipynb'
    )

    null_category_count = products['category'].isna().sum()
    check(
        'product_master',
        'category: zero null values (required for category-level analysis)',
        null_category_count == 0,
        str(null_category_count) + ' null category values found'
    )

    # NOTE: launch_date null rate of 62% is expected from raw data
    null_launch_date_pct = products['launch_date'].isna().mean() * 100
    check(
        'product_master',
        'launch_date: null percentage is 65% or less (62% was unparseable in raw data, this is expected)',
        null_launch_date_pct <= 65,
        str(round(null_launch_date_pct, 1)) + '% are null. Higher than expected.',
        severity='WARN'
    )

    print("")
    print("  Summary:")
    print("  Rows: " + str(len(products)))
    print("  Unique SKUs: " + str(products['sku'].nunique()))
    print("  Categories: " + str(products['category'].value_counts().to_dict()))
    print("  launch_date null: " + str(round(null_launch_date_pct, 1)) + "% (62% is expected based on raw data audit)")


CHECK GROUP 7: PRODUCT_MASTER
SKU format, no duplicates, Title Case, launch_date null rate.
  PASS   sku: uppercase, no whitespace, no duplicates (dimension table key)
  PASS   product_name: no ALL CAPS values (Title Case applied)
  PASS   category: zero null values (required for category-level analysis)
  PASS   launch_date: null percentage is 65% or less (62% was unparseable in raw data, this is expected)

  Summary:
  Rows: 80000
  Unique SKUs: 80000
  Categories: {'Drink Mix': 16098, 'Coffee': 16091, 'Bundle': 15978, 'Snack': 15946, 'Gift Set': 15887}
  launch_date null: 62.3% (62% is expected based on raw data audit)


## purchase_orders table

In [ ]:
section_header(
    "CHECK: PURCHASE_ORDERS",
    "total_po_cost formula, unit_cost constraints, status values."
)

if po is not None:

    nat_date_count = pd.to_datetime(po['created_date'], errors='coerce').isna().sum()
    check(
        'purchase_orders',
        'created_date: zero NaT values',
        nat_date_count == 0,
        str(nat_date_count) + ' NaT values in created_date'
    )

    sku_has_lowercase = po['sku'].dropna().str.contains('[a-z]').any()
    check(
        'purchase_orders',
        'sku: all values are uppercase (JOIN key)',
        not sku_has_lowercase,
        'Some SKU values still contain lowercase letters'
    )

    # CHECK: unit_cost has no negative values
    # WHAT: verify that .abs() was applied to negative unit_cost values
    # WHY:  negative unit_cost produces a negative total_po_cost, which makes
    #       inventory value calculations incorrect.
    #       A PO with negative cost looks like a refund from the supplier.
    neg_unit_cost_count = (po['unit_cost'] < 0).sum()
    check(
        'purchase_orders',
        'unit_cost: no negative values (abs() was applied in cleaning)',
        neg_unit_cost_count == 0,
        str(neg_unit_cost_count) + ' rows still have negative unit_cost. '
        'Fix: re-run Cell 13 in 02_data_cleaning.ipynb'
    )

    neg_qty_count = (po['qty_ordered'].fillna(0) < 0).sum()
    check(
        'purchase_orders',
        'qty_ordered: no negative values',
        neg_qty_count == 0,
        str(neg_qty_count) + ' rows have negative qty_ordered'
    )

    # CHECK: total_po_cost formula is correct
    # WHAT: verify total_po_cost = qty_ordered * unit_cost on every row
    # WHY:  total_po_cost is used for inventory valuation and supplier spend analysis.
    #       A formula error would make dead stock costs incorrect.
    rows_with_po_values = po.dropna(subset=['qty_ordered', 'unit_cost', 'total_po_cost'])
    expected_po_cost    = rows_with_po_values['qty_ordered'] * rows_with_po_values['unit_cost']
    formula_mismatch_count = (rows_with_po_values['total_po_cost'] - expected_po_cost).abs().gt(0.01).sum()
    check(
        'purchase_orders',
        'total_po_cost equals qty_ordered times unit_cost on every row',
        formula_mismatch_count == 0,
        str(formula_mismatch_count) + ' rows have total_po_cost that does not match the formula'
    )

    expected_po_statuses = {'Received', 'Cancelled', 'Open', 'Partially Received'}
    actual_po_statuses   = set(po['status'].dropna().unique())
    unexpected_statuses  = actual_po_statuses - expected_po_statuses
    check(
        'purchase_orders',
        'status: only known values in Title Case',
        len(unexpected_statuses) == 0,
        'Unexpected status values: ' + str(unexpected_statuses)
    )

    remaining_raw_cols = [c for c in po.columns if c.endswith('_raw')]
    check(
        'purchase_orders',
        'no _raw columns remaining in clean file',
        len(remaining_raw_cols) == 0,
        'These _raw columns were not dropped: ' + str(remaining_raw_cols)
    )

    print("")
    print("  Summary:")
    print("  Rows: " + str(len(po)))
    print("  Status breakdown: " + str(po['status'].value_counts().to_dict()))


CHECK: PURCHASE_ORDERS
total_po_cost formula, unit_cost constraints, status values.
  PASS   created_date: zero NaT values
  PASS   sku: all values are uppercase (JOIN key)
  PASS   unit_cost: no negative values (abs() was applied in cleaning)
  PASS   qty_ordered: no negative values
  PASS   total_po_cost equals qty_ordered times unit_cost on every row
  PASS   status: only known values in Title Case
  PASS   no _raw columns remaining in clean file

  Summary:
  Rows: 90000
  Status breakdown: {'Received': 22601, 'Cancelled': 22522, 'Open': 22459, 'Partially Received': 22418}


## promotions_log table

In [ ]:
section_header(
    "CHECK: PROMOTIONS_LOG",
    "Date range logic, promo_code format, discount percentage range."
)

if promos is not None:

    nat_start_count = pd.to_datetime(promos['start_date'], errors='coerce').isna().sum()
    check(
        'promotions_log',
        'start_date: zero NaT values',
        nat_start_count == 0,
        str(nat_start_count) + ' NaT values in start_date'
    )

    # CHECK: end_date is not earlier than start_date
    # WHAT: a sanity check, not a fix
    # WHY:  a promotion cannot logically end before it starts.
    #       These rows are a data quality problem that needs business review.
    #       We flag them as WARN because they were documented in cleaning.
    promos_copy = promos.copy()
    promos_copy['start_date'] = pd.to_datetime(promos_copy['start_date'], errors='coerce')
    promos_copy['end_date']   = pd.to_datetime(promos_copy['end_date'],   errors='coerce')
    both_dates_valid   = promos_copy['start_date'].notna() & promos_copy['end_date'].notna()
    backwards_date_count = (promos_copy.loc[both_dates_valid, 'end_date'] < promos_copy.loc[both_dates_valid, 'start_date']).sum()
    check(
        'promotions_log',
        'end_date is not earlier than start_date on any row',
        backwards_date_count == 0,
        str(backwards_date_count) + ' rows have end_date earlier than start_date. '
        'This is a raw data problem, documented in the cleaning log.',
        severity='WARN'
    )

    promo_code_has_lowercase  = promos['promo_code'].dropna().str.contains('[a-z]').any()
    promo_code_has_whitespace = promos['promo_code'].dropna().str.contains(r'\s', regex=True).any()
    check(
        'promotions_log',
        'promo_code: uppercase with no whitespace (JOIN key to orders.promo_code)',
        not promo_code_has_lowercase and not promo_code_has_whitespace,
        'lowercase: ' + str(promo_code_has_lowercase) + ', whitespace: ' + str(promo_code_has_whitespace)
    )

    # CHECK: discount_pct is between 0 and 100
    # WHAT: verify that no impossible discount percentages exist
    # WHY:  a discount of 150% means the business pays the customer $1.50 per $1.00 purchased.
    #       A discount of -10% means a markup, not a discount.
    #       Both are almost certainly data entry errors.
    valid_pct_values       = promos['discount_pct'].dropna()
    out_of_range_pct_count = ((valid_pct_values < 0) | (valid_pct_values > 100)).sum()
    check(
        'promotions_log',
        'discount_pct: all values are between 0 and 100',
        out_of_range_pct_count == 0,
        str(out_of_range_pct_count) + ' values are outside the 0 to 100 range'
    )

    if orders is not None:
        promo_codes_in_orders = set(orders['promo_code'].dropna().unique()) - {'NONE'}
        promo_codes_in_log    = set(promos['promo_code'].dropna().unique())
        unmatched_promo_codes = promo_codes_in_orders - promo_codes_in_log
        check(
            'promotions_log',
            'all promo codes used in orders also exist in promotions_log',
            len(unmatched_promo_codes) == 0,
            'Promo codes in orders but not in promotions_log: ' + str(unmatched_promo_codes),
            severity='WARN'
        )

    remaining_raw_cols = [c for c in promos.columns if c.endswith('_raw')]
    check(
        'promotions_log',
        'no _raw columns remaining in clean file',
        len(remaining_raw_cols) == 0,
        'These _raw columns were not dropped: ' + str(remaining_raw_cols)
    )

    print("")
    print("  Summary:")
    print("  Rows: " + str(len(promos)))
    print("  Unique promo codes: " + str(promos['promo_code'].nunique()))
    print("  Average discount: " + str(round(promos['discount_pct'].mean(), 1)) + "%")



CHECK: PROMOTIONS_LOG
Date range logic, promo_code format, discount percentage range.
  PASS   start_date: zero NaT values
  PASS   end_date is not earlier than start_date on any row
  PASS   promo_code: uppercase with no whitespace (JOIN key to orders.promo_code)
  PASS   discount_pct: all values are between 0 and 100
  PASS   all promo codes used in orders also exist in promotions_log
  PASS   no _raw columns remaining in clean file

  Summary:
  Rows: 85000
  Unique promo codes: 5
  Average discount: 15.0%


# marketing_spend table

In [ ]:
section_header(
    "CHECK: MARKETING_SPEND",
    "Spend and clicks constraints. Anomaly status documented."
)

if mkt is not None:

    nat_date_count = pd.to_datetime(mkt['log_timestamp'], errors='coerce').isna().sum()
    check(
        'marketing_spend',
        'log_date: zero NaT values',
        nat_date_count == 0,
        str(nat_date_count) + ' NaT values in log_date'
    )

    neg_spend_count = (mkt['spend'] < 0).sum()
    check(
        'marketing_spend',
        'spend: no negative values (clipped to 0)',
        neg_spend_count == 0,
        str(neg_spend_count) + ' rows still have negative spend'
    )

    neg_clicks_count = (mkt['clicks'].fillna(0) < 0).sum()
    check(
        'marketing_spend',
        'clicks: no negative values',
        neg_clicks_count == 0,
        str(neg_clicks_count) + ' rows have negative clicks'
    )

    # This check always passes (it is purely a WARN to document the anomaly)
    total_spend = mkt['spend'].sum()
    check(
        'marketing_spend',
        'ANOMALY DOCUMENTED: total spend is much larger than total revenue, excluded from ROAS',
        True,
        'Total spend: $' + str(round(total_spend)) + '. '
        'Cause unknown. Awaiting Finance clarification. '
        'CPC comparisons between platforms are still valid.',
        severity='WARN'
    )
    print("         Total spend = $" + str(round(total_spend)) + ". Finance clarification needed before ROAS analysis.")

    remaining_raw_cols = [c for c in mkt.columns if c.endswith('_raw')]
    check(
        'marketing_spend',
        'no _raw columns remaining in clean file',
        len(remaining_raw_cols) == 0,
        'These _raw columns were not dropped: ' + str(remaining_raw_cols)
    )

    print("")
    print("  Summary:")
    print("  Rows: " + str(len(mkt)))
    print("  Platforms: " + str(mkt['ad_platform'].value_counts().to_dict()))


CHECK: MARKETING_SPEND
Spend and clicks constraints. Anomaly status documented.
  PASS   log_date: zero NaT values
  PASS   spend: no negative values (clipped to 0)
  PASS   clicks: no negative values
  PASS   ANOMALY DOCUMENTED: total spend is much larger than total revenue, excluded from ROAS
         Total spend = $305185119. Finance clarification needed before ROAS analysis.
  PASS   no _raw columns remaining in clean file

  Summary:
  Rows: 90000
  Platforms: {'Google Ads': 22869, 'Instagram Ads': 22413, 'Tiktok Ads': 22402, 'Facebook Ads': 22316}


## Cross-table referential integrity checks
###### These checks verify that JOIN keys are consistent across tables.
###### If any of these fail, JOIN-based analysis will produce wrong results.
###### A FAIL here means some rows will be silently dropped or duplicated in JOINs.

In [ ]:
section_header(
    "CHECK: CROSS-TABLE REFERENTIAL INTEGRITY",
    "JOIN key consistency across tables. Failures here break all JOIN-based analysis."
)

# CHECK: no orphan order_item rows
# WHAT: every order_id in order_items must exist in orders
# WHY:  if an order_item has an order_id that is not in orders, that row
#       cannot be joined to order-level data (platform, customer, date).
#       It becomes an orphan record that skews item-level metrics.
if orders is not None and items is not None:
    orphan_item_count = (~items['order_id'].isin(orders['order_id'])).sum()
    check(
        'cross-table',
        'order_items.order_id -> orders: no orphan lines',
        orphan_item_count == 0,
        str(orphan_item_count) + ' order_item rows have order_ids not found in orders',
        severity='WARN'
    )

# CHECK: no orphan refund rows
# WHAT: every order_id in refunds must exist in orders
# WHY:  orphan refunds cannot be linked to a specific order, platform, or customer.
#       They make refund-by-platform and refund-by-customer analysis incorrect.
if orders is not None and refunds is not None:
    orphan_refund_count = (~refunds['order_id'].isin(orders['order_id'])).sum()
    check(
        'cross-table',
        'refunds.order_id -> orders: no orphan refunds',
        orphan_refund_count == 0,
        str(orphan_refund_count) + ' refund rows have order_ids not found in orders',
        severity='WARN'
    )

# CHECK: all SKUs in order_items exist in product_master
# WHAT: every SKU sold must have a matching product record
# WHY:  without a product_master row, you cannot get category or product_name.
#       Category-level revenue reports will be incomplete.
if items is not None and products is not None:
    sku_not_in_products_count = (~items['sku'].dropna().isin(products['sku'])).sum()
    check(
        'cross-table',
        'order_items.sku -> product_master: all SKUs have a matching product record',
        sku_not_in_products_count == 0,
        str(sku_not_in_products_count) + ' SKUs in order_items have no matching row in product_master',
        severity='WARN'
    )

# CHECK: COGS coverage is at least 90%
# WHAT: at least 90% of order_item SKUs must have a matching cost in cost_history
# WHY:  without cost data, gross margin cannot be calculated for those rows.
#       If 27% of rows have no cost, gross margin reports will be based on only 73% of sales.
if items is not None and costs is not None:
    sku_not_in_costs_count = (~items['sku'].dropna().isin(costs['sku'])).sum()
    sku_not_in_costs_pct   = sku_not_in_costs_count / len(items) * 100
    check(
        'cross-table',
        'order_items.sku -> cost_history: at least 90% of order lines have a COGS record',
        sku_not_in_costs_pct <= 10,
        str(round(sku_not_in_costs_pct, 1)) + '% of order lines have no matching cost. '
        'Gross margin calculations will be incomplete for these rows.',
        severity='WARN'
    )

# CHECK: customer profile coverage is at least 90%
# WHAT: at least 90% of order emails must match a customer_profiles row
# WHY:  without a profile, you cannot segment that customer (New, VIP, Churn Risk, etc.).
#       Customer segment analysis will be based on an incomplete set of orders.
if orders is not None and customers is not None:
    orders_with_no_profile_count = (~orders['customer_email'].isin(customers['customer_email'])).sum()
    orders_with_no_profile_pct   = orders_with_no_profile_count / len(orders) * 100
    check(
        'cross-table',
        'orders.customer_email -> customer_profiles: at least 90% of orders have a customer profile',
        orders_with_no_profile_pct <= 10,
        str(round(orders_with_no_profile_pct, 1)) + '% of orders have no matching customer profile',
        severity='WARN'
    )

# CHECK: all purchase_order SKUs exist in product_master
# WHAT: every SKU ordered from a supplier should have a product record
# WHY:  without a product record, you cannot categorize inventory by product type.
if po is not None and products is not None:
    po_sku_notnull = po['sku'].dropna()          # loại null ra
    po_null_count  = po['sku'].isna().sum()       # đếm null riêng
    po_not_in_prod = (~po_sku_notnull.isin(products['sku'])).sum()  # = 0
    check(
        'cross-table',
        'purchase_orders.sku -> product_master: all SKUs have a matching product record',
        po_sku_not_in_products_count == 0,
        str(po_sku_not_in_products_count) + ' PO SKUs have no matching row in product_master',
        severity='WARN'
    )


CHECK: CROSS-TABLE REFERENTIAL INTEGRITY
JOIN key consistency across tables. Failures here break all JOIN-based analysis.
  PASS   order_items.order_id -> orders: no orphan lines
  PASS   refunds.order_id -> orders: no orphan refunds
  PASS   order_items.sku -> product_master: all SKUs have a matching product record
  WARN   order_items.sku -> cost_history: at least 90% of order lines have a COGS record
         27.6% of order lines have no matching cost. Gross margin calculations will be incomplete for these rows.
  PASS   orders.customer_email -> customer_profiles: at least 90% of orders have a customer profile
  WARN   purchase_orders.sku -> product_master: all SKUs have a matching product record
         3581 PO SKUs have no matching row in product_master


##  Final summary and save results

In [ ]:
results_df = pd.DataFrame(all_qc_results)

pass_count = (results_df['status'] == 'PASS').sum()
fail_count = (results_df['status'] == 'FAIL').sum()
warn_count = (results_df['status'] == 'WARN').sum()
total_count = len(results_df)

print("")
print("=" * 60)
print("NOTEBOOK 03: QUALITY CHECK FINAL REPORT")
print("Completed: " + datetime.now().strftime('%Y-%m-%d %H:%M'))
print("=" * 60)
print("")
print("Total checks: " + str(total_count))
print("PASS:         " + str(pass_count))
print("FAIL:         " + str(fail_count) + ("  <-- BLOCKING: fix before BigQuery upload" if fail_count > 0 else "  <-- none, all clean"))
print("WARN:         " + str(warn_count) + "  <-- known issues, document in README")

if fail_count > 0:
    print("")
    print("-" * 60)
    print("FAILED CHECKS - fix these in 02_data_cleaning.ipynb:")
    print("-" * 60)
    failed_rows = results_df[results_df['status'] == 'FAIL']
    for index, row in failed_rows.iterrows():
        print("")
        print("  Table: " + row['table'])
        print("  Check: " + row['rule'])
        print("  Issue: " + row['detail'])

if warn_count > 0:
    print("")
    print("-" * 60)
    print("WARNINGS - document these in your README and data dictionary:")
    print("-" * 60)
    warn_rows = results_df[results_df['status'] == 'WARN']
    for index, row in warn_rows.iterrows():
        print("  [" + row['table'] + "] " + row['rule'])
        if row['detail']:
            print("    -> " + row['detail'])

qc_output_path = os.path.join(AUDIT_DIR, '03_qc_results.csv')
results_df.to_csv(qc_output_path, index=False)

print("")
print("-" * 60)
print("QC results saved to: " + qc_output_path)
print("-" * 60)

print("")
print("=" * 60)
if fail_count == 0:
    print("RESULT: ALL CHECKS PASSED")
    print("The data in datasets_clean/ is ready for BigQuery upload.")
    print("Next step: run 04_bigquery_upload.ipynb")
else:
    print("RESULT: " + str(fail_count) + " CHECKS FAILED")
    print("Steps to fix:")
    print("  1. Open 02_data_cleaning.ipynb")
    print("  2. Find the cell for the failing table")
    print("  3. Fix the issue described above")
    print("  4. Re-run that single cell only")
    print("  5. Come back here and re-run this entire notebook")
print("=" * 60)

print("")
print("All files produced by this pipeline:")
print("  audit/01_audit_report.json      raw data profile from Notebook 01")
print("  audit/01_audit_summary.txt      human-readable audit summary")
print("  audit/02_cleaning_log.txt       timestamped log of every change made")
print("  audit/02_cleaning_summary.json  row counts and null% after cleaning")
print("  audit/03_qc_results.csv         pass/fail results from this notebook")
print("  datasets_clean/                 11 clean CSV files, ready for BigQuery")